In [191]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [192]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [193]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model



# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase2"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [194]:
TEXT_FILES = [
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold2-text.json"
]
GOLD_FILES = [
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-gold.json"
]
LABELS_FILE =[
"Phase_2_LNER/Train/labels.txt"
]
TEST_FILE =["Phase_2_LNER/Test/fold3-text.json"]

In [195]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [196]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/rogue2222/tokenizer-results/project_bpe_16k.pkl"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    bpe = pickle.load(f)
print("BPE vocab size:", len(bpe.itos))


BPE vocab size: 7999


In [197]:
import json

TEXT_FILES = [
    "/kaggle/input/datasets/rogue2222/phase2-dataset/fold1-text.json",
    "/kaggle/input/datasets/rogue2222/phase2-dataset/fold2-text.json"
]

GOLD_FILES = [
    "/kaggle/input/datasets/rogue2222/phase2-dataset/fold1-gold.json",
    "/kaggle/input/datasets/rogue2222/phase2-dataset/fold2-gold.json"
]

LABELS_FILE = "/kaggle/input/datasets/rogue2222/phase2-dataset/labels.txt"

TEST_FILE = "/kaggle/input/datasets/rogue2222/phase2-dataset/fold3-text.json"


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# -------------------------
# Load training text
# -------------------------
fold1_text = load_json(TEXT_FILES[0])
fold2_text = load_json(TEXT_FILES[1])

# -------------------------
# Load training gold
# -------------------------
fold1_gold = load_json(GOLD_FILES[0])
fold2_gold = load_json(GOLD_FILES[1])

# -------------------------
# Load labels
# -------------------------
with open(LABELS_FILE, "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

# -------------------------
# Load test text
# -------------------------
fold3_text = load_json(TEST_FILE)

In [198]:
train_text = {
    **fold1_text,
    **fold2_text
}

train_gold = {
    **fold1_gold,
    **fold2_gold
}
print("Fold 1 text:", len(fold1_text))
print("Fold 1 gold:", len(fold1_gold))

print("Fold 2 text:", len(fold2_text))
print("Fold 2 gold:", len(fold2_gold))

print("Total training documents:", len(train_text))
print("Total gold documents:", len(train_gold))

print("Test documents:", len(fold3_text))

print("Labels:")
print(labels)


Fold 1 text: 35
Fold 1 gold: 35
Fold 2 text: 35
Fold 2 gold: 35
Total training documents: 70
Total gold documents: 70
Test documents: 35
Labels:
['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [199]:
print("Checking Text/gold document IDs")
assert set(fold1_text.keys()) == set(fold1_gold.keys())
assert set(fold2_text.keys()) == set(fold2_gold.keys())
assert set(train_text.keys()) == set(train_gold.keys())



Checking Text/gold document IDs


In [200]:
doc_id = next(iter(train_text))
print(train_text[doc_id][:1000])
print(train_gold[doc_id])
print(labels)

REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, Jage Ram (A-1) and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis respectively, entered the land where the complainant was working with his sons and asked

In [201]:
train_text[doc_id][137:153] == "STATE OF HARYANA"

True

In [202]:
converter = ToTokensConverter(bpe.itos)
tokenizer = bpe

In [203]:
def encode_with_offsets( text):
    token_ids = []
    offsets = []
    i = 0
    while i < len(text):
        # Skip whitespace
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        word_start = i
        while i < len(text) and not text[i].isspace():
            i += 1
        word_end = i
        word = text[word_start:word_end]
        word_bytes = word.encode("utf-8")
        ids = converter._tokenize_word(
            word_bytes + b"</w>"
        )
        # Build UTF-8 byte -> character mapping
        byte_to_char = {}
        byte_pos = 0
        for char_idx, ch in enumerate(word):
            char_bytes = ch.encode("utf-8")
            n = len(char_bytes)
            for j in range(n):
                byte_to_char[byte_pos + j] = char_idx
            byte_pos += n
            # Boundary after character
            byte_to_char[byte_pos] = char_idx + 1
        # Process BPE tokens
        byte_pos = 0
        for token_id in ids:
            token_bytes = bpe.itos[token_id]
            # Standalone </w>
            if token_bytes == b"</w>":
                continue
            # Remove </w> if attached
            piece_bytes = token_bytes
            if piece_bytes.endswith(b"</w>"):
                piece_bytes = piece_bytes[:-4]
            if not piece_bytes:
                continue
            byte_start = byte_pos
            byte_end = byte_pos + len(piece_bytes)
            if byte_end > len(word_bytes):
                raise ValueError(
                    f"Invalid BPE byte range: "
                    f"{byte_start}:{byte_end}, "
                    f"word has {len(word_bytes)} bytes. "
                    f"word={word!r}, "
                    f"token={token_bytes!r}"
                )
            # ----------------------------------------------------
            # Find character containing the START byte
            # ----------------------------------------------------
            char_start_local = byte_to_char[byte_start]
            # ----------------------------------------------------
            # Find character containing the END byte.
            #
            # byte_end is exclusive.
            #
            # If byte_end is a character boundary:
            #     use that boundary.
            #
            # If byte_end falls inside a character:
            #     include the entire character.
            # ----------------------------------------------------
            if byte_end in byte_to_char:
                char_end_local = byte_to_char[byte_end]
            else:
                # byte_end is inside a UTF-8 character.
                # Find the character containing byte_end.
                char_end_local = byte_to_char[byte_end - 1] + 1
            # Make sure at least one character is covered.
            if char_end_local <= char_start_local:
                char_end_local = char_start_local + 1
            # Convert local word offsets to global text offsets.
            char_start = word_start + char_start_local
            char_end = word_start + char_end_local
            token_ids.append(token_id)
            offsets.append((char_start, char_end))
            byte_pos = byte_end
    return token_ids, offsets
    
doc_id = "37849282"

text = train_text[doc_id]

token_ids, offsets = encode_with_offsets(text)

print("Number of tokens:", len(token_ids))
print("Number of offsets:", len(offsets))

for i, (tid, offset) in enumerate(zip(token_ids, offsets)):

    if 180 <= offset[0] <= 195:

        print(
            i,
            tid,
            repr(tokenizer.itos[tid]),
            offset,
            repr(text[offset[0]:offset[1]])
        )

Number of tokens: 5161
Number of offsets: 5161
77 7747 b'(I' (180, 182) '(I'
78 315 b')</w>' (182, 183) ')'
79 80 b'L' (184, 185) 'L'
80 88 b'T' (185, 186) 'T'
81 3200 b'D.</w>' (186, 188) 'D.'
82 977 b'\xe2\x80' (189, 190) '…'
83 170 b'\xa6' (189, 190) '…'
84 6641 b'APPELL' (191, 197) 'APPELL'


In [204]:
def test_offsets(text_data, max_errors=20):

    total_docs = 0
    total_tokens = 0
    errors = []

    for doc_id, text in text_data.items():

        total_docs += 1

        try:
            token_ids, offsets = encode_with_offsets(text)

            if len(token_ids) != len(offsets):
                errors.append(
                    (doc_id, "token_ids and offsets length mismatch")
                )
                continue

            total_tokens += len(token_ids)

            # Check every offset
            for token_id, (start, end) in zip(token_ids, offsets):

                # Valid range
                if not (0 <= start < end <= len(text)):
                    errors.append(
                        (
                            doc_id,
                            f"Invalid offset: {(start, end)}, "
                            f"text length={len(text)}"
                        )
                    )

                    if len(errors) >= max_errors:
                        break

                    continue

                # Extract original text
                piece = text[start:end]

                if len(piece) == 0:
                    errors.append(
                        (doc_id, f"Empty piece at {(start, end)}")
                    )

                    if len(errors) >= max_errors:
                        break

        except Exception as e:

            errors.append(
                (doc_id, repr(e))
            )

        if len(errors) >= max_errors:
            break

    print("Documents tested:", total_docs)
    print("Total tokens:", total_tokens)
    print("Errors:", len(errors))

    if errors:
        print("\nFirst errors:")
        for error in errors[:max_errors]:
            print(error)
    else:
        print("\n✓ All offsets are valid")

In [205]:
import pandas as pd
from collections import Counter, defaultdict


def analyze_gold_token_alignment(
    train_text,
    train_gold,
    tokenizer,
    verbose=True
):
    """
    Analyze how gold character-level entity spans align with BPE token spans.

    For every gold entity:
        - finds all overlapping BPE tokens
        - computes the covering token span
        - compares covering token span with gold character span
        - captures prefix/suffix mismatches
        - stores detailed information for later inspection

    Returns
    -------
    summary : dict
        Overall statistics.

    entity_stats : pd.DataFrame
        Statistics grouped by entity label.

    mismatches_df : pd.DataFrame
        One row for every non-exact alignment.

    all_df : pd.DataFrame
        One row for every gold entity.
    """

    records = []

    for doc_idx, (doc_id, text) in enumerate(train_text.items()):

        if doc_id not in train_gold:
            continue

        gold_annotations = train_gold[doc_id]

        # Tokenize document and get character offsets
        token_ids, offsets = encode_with_offsets(text)

        for entity_idx, annotation in enumerate(gold_annotations):

            gold_start, gold_end, gold_label, gold_text = annotation

            # ---------------------------------------------------------
            # Find tokens overlapping the gold entity
            # ---------------------------------------------------------
            overlapping = []

            for token_idx, (tok_start, tok_end) in enumerate(offsets):

                overlap_start = max(tok_start, gold_start)
                overlap_end = min(tok_end, gold_end)

                if overlap_start < overlap_end:
                    overlapping.append(
                        (token_idx, tok_start, tok_end)
                    )

            # ---------------------------------------------------------
            # No token matched
            # ---------------------------------------------------------
            if not overlapping:

                records.append({
                    "doc_id": doc_id,
                    "entity_idx": entity_idx,
                    "label": gold_label,
                    "gold_text": gold_text,

                    "gold_start": gold_start,
                    "gold_end": gold_end,

                    "first_token_idx": None,
                    "last_token_idx": None,

                    "token_start": None,
                    "token_end": None,

                    "start_diff": None,
                    "end_diff": None,

                    "prefix_extra": "",
                    "suffix_extra": "",

                    "token_text": "",

                    "alignment_type": "NO_TOKEN"
                })

                continue

            # ---------------------------------------------------------
            # Covering token span
            # ---------------------------------------------------------
            first_token_idx = overlapping[0][0]
            last_token_idx = overlapping[-1][0]

            token_start = offsets[first_token_idx][0]
            token_end = offsets[last_token_idx][1]

            # ---------------------------------------------------------
            # Differences
            #
            # positive start_diff:
            #     token starts AFTER gold
            #
            # negative start_diff:
            #     token starts BEFORE gold
            #
            # positive end_diff:
            #     token ends AFTER gold
            #
            # negative end_diff:
            #     token ends BEFORE gold
            # ---------------------------------------------------------
            start_diff = token_start - gold_start
            end_diff = token_end - gold_end

            # ---------------------------------------------------------
            # Extra text before gold
            # ---------------------------------------------------------
            prefix_extra = ""

            if token_start < gold_start:
                prefix_extra = text[token_start:gold_start]

            # ---------------------------------------------------------
            # Extra text after gold
            # ---------------------------------------------------------
            suffix_extra = ""

            if token_end > gold_end:
                suffix_extra = text[gold_end:token_end]

            # ---------------------------------------------------------
            # Complete text covered by BPE tokens
            # ---------------------------------------------------------
            token_text = text[token_start:token_end]

            # ---------------------------------------------------------
            # Classify alignment
            # ---------------------------------------------------------
            if token_start == gold_start and token_end == gold_end:

                alignment_type = "EXACT"

            elif token_start == gold_start and token_end > gold_end:

                alignment_type = "SUFFIX_ONLY"

            elif token_start < gold_start and token_end == gold_end:

                alignment_type = "PREFIX_ONLY"

            elif token_start < gold_start and token_end > gold_end:

                alignment_type = "PREFIX_AND_SUFFIX"

            elif token_start > gold_start and token_end < gold_end:

                alignment_type = "INSIDE_GOLD"

            elif token_start > gold_start:

                alignment_type = "START_MISMATCH"

            elif token_end < gold_end:

                alignment_type = "END_MISMATCH"

            else:

                alignment_type = "OTHER"

            records.append({
                "doc_id": doc_id,
                "entity_idx": entity_idx,
                "label": gold_label,
                "gold_text": gold_text,

                "gold_start": gold_start,
                "gold_end": gold_end,

                "first_token_idx": first_token_idx,
                "last_token_idx": last_token_idx,

                "token_start": token_start,
                "token_end": token_end,

                "start_diff": start_diff,
                "end_diff": end_diff,

                "prefix_extra": prefix_extra,
                "suffix_extra": suffix_extra,

                "token_text": token_text,

                "alignment_type": alignment_type
            })

        if verbose and (doc_idx + 1) % 100 == 0:
            print(f"Processed {doc_idx + 1} documents...")

    # ================================================================
    # DataFrame
    # ================================================================

    all_df = pd.DataFrame(records)

    # ================================================================
    # Overall statistics
    # ================================================================

    total = len(all_df)

    alignment_counts = (
        all_df["alignment_type"]
        .value_counts()
        .to_dict()
    )

    exact_count = alignment_counts.get("EXACT", 0)

    mismatch_count = total - exact_count

    summary = {
        "total_gold_entities": total,
        "exact": exact_count,
        "mismatched": mismatch_count,
        "exact_percentage": (
            100 * exact_count / total
            if total > 0 else 0
        ),
        "mismatch_percentage": (
            100 * mismatch_count / total
            if total > 0 else 0
        ),
        "alignment_counts": alignment_counts
    }

    # ================================================================
    # Statistics by entity type
    # ================================================================

    entity_stats = (
        all_df
        .groupby("label")
        .agg(
            total=("label", "size"),
            exact=("alignment_type",
                   lambda x: (x == "EXACT").sum()),
            suffix_only=("alignment_type",
                         lambda x: (x == "SUFFIX_ONLY").sum()),
            prefix_only=("alignment_type",
                         lambda x: (x == "PREFIX_ONLY").sum()),
            prefix_suffix=("alignment_type",
                           lambda x: (x == "PREFIX_AND_SUFFIX").sum()),
            other=("alignment_type",
                   lambda x: (x != "EXACT").sum())
        )
        .reset_index()
    )

    entity_stats["exact_pct"] = (
        100 * entity_stats["exact"] /
        entity_stats["total"]
    )

    entity_stats["mismatch_pct"] = (
        100 * entity_stats["other"] /
        entity_stats["total"]
    )

    entity_stats = entity_stats.sort_values(
        "mismatch_pct",
        ascending=False
    )

    # ================================================================
    # Mismatches only
    # ================================================================

    mismatches_df = all_df[
        all_df["alignment_type"] != "EXACT"
    ].copy()

    return (
        summary,
        entity_stats,
        mismatches_df,
        all_df
    )

In [206]:
summary, entity_stats, mismatches_df, all_df = (
    analyze_gold_token_alignment(
        train_text=train_text,
        train_gold=train_gold,
        tokenizer=converter,
        verbose=True
    )
)

In [207]:
print("\n========== OVERALL ALIGNMENT ==========")

print(
    f"Total gold entities : {summary['total_gold_entities']}"
)

print(
    f"Exact               : {summary['exact']}"
)

print(
    f"Mismatched           : {summary['mismatched']}"
)

print(
    f"Exact %              : {summary['exact_percentage']:.2f}%"
)

print(
    f"Mismatch %           : {summary['mismatch_percentage']:.2f}%"
)

print("\nAlignment types:")

for k, v in summary["alignment_counts"].items():
    pct = 100 * v / summary["total_gold_entities"]

    print(
        f"{k:20s} "
        f"{v:7d} "
        f"({pct:6.2f}%)"
    )


========== OVERALL ALIGNMENT ==========
Total gold entities : 8393
Exact               : 6082
Mismatched           : 2311
Exact %              : 72.47%
Mismatch %           : 27.53%

Alignment types:
EXACT                   6082 ( 72.47%)
SUFFIX_ONLY             2239 ( 26.68%)
PREFIX_ONLY               42 (  0.50%)
END_MISMATCH              16 (  0.19%)
PREFIX_AND_SUFFIX         14 (  0.17%)


In [208]:
# end_df = mismatches_df[
#     mismatches_df["alignment_type"] == "END_MISMATCH"
# ].copy()
# end_df.head()
# end_counts = (
#     end_df["end_diff"]
#     .value_counts()
#     .rename_axis("endfix")
#     .reset_index(name="count")
# )

# end_counts["percentage"] = (
#     100 * end_counts["count"] / len(end_df)
# )

# display(end_counts.head(50))

In [209]:
# row = end_df.iloc[0]
# text = train_text["76640285"]

# gold = train_text[row["gold_start"]:row["gold_end"]]
# token = train_text[row["token_start"]:row["token_end"]]

# print("Gold repr: ", repr(gold))
# print("Token repr:", repr(token))
# print("Gold len:  ", len(gold))
# print("Token len: ", len(token))
# print("Next char: ", repr(text[row["token_end"]]))
# print("Gold end exclusive:", row["gold_end"])
# print("Token end exclusive:", row["token_end"])

In [210]:
# display(
#     end_df[
#         [
#             "doc_id",
#             "label",
#             "gold_text",
#             "token_text",
#             "suffix_extra",
#             "gold_start",
#             "gold_end",
#             "token_start",
#             "token_end",
#             "end_diff"
#         ]
#     ].head(100)
# )

In [211]:
from collections import defaultdict, Counter

prefix_map_counts = defaultdict(Counter)

prefix_df = mismatches_df[
    mismatches_df["alignment_type"] == "PREFIX_ONLY"
].copy()

for _, row in prefix_df.iterrows():
    key = (row["label"], row["token_text"])
    prefix_map_counts[key][row["gold_text"]] += 1

prefix_map = {}

for key, counter in prefix_map_counts.items():
    if len(counter) == 1:
        prefix_map[key] = next(iter(counter))

print("Number of mappings:", len(prefix_map))
print("Total PREFIX_ONLY cases:", len(prefix_df))
print("Unique mappings:", len(prefix_map))

mapped_count = sum(
    (row["label"], row["token_text"]) in prefix_map
    for _, row in prefix_df.iterrows()
)

print("Cases covered:", mapped_count)
print("Coverage: {:.2f}%".format(
    100 * mapped_count / max(len(prefix_df), 1)
))

Number of mappings: 34
Total PREFIX_ONLY cases: 42
Unique mappings: 34
Cases covered: 42
Coverage: 100.00%


In [212]:
# Before correction
before_exact = (
    mismatches_df["gold_text"].str.strip()
    == mismatches_df["token_text"].str.strip()
)

print("BEFORE CORRECTION")
print("Exact matches:", before_exact.sum())
print("Total:", len(mismatches_df))
print(f"Exact match rate: {before_exact.mean() * 100:.2f}%")

# After correction
corrected_df = mismatches_df.copy()

def correct_prefix(row):
    key = (row["label"], row["token_text"])

    if (
        row["alignment_type"] == "PREFIX_ONLY"
        and key in prefix_map
    ):
        return prefix_map[key]

    return row["token_text"]

corrected_df["corrected_text"] = corrected_df.apply(
    correct_prefix,
    axis=1
)

after_exact = (
    corrected_df["gold_text"].str.strip()
    == corrected_df["corrected_text"].str.strip()
)

print("\nAFTER CORRECTION")
print("Exact matches:", after_exact.sum())
print("Total:", len(corrected_df))
print(f"Exact match rate: {after_exact.mean() * 100:.2f}%")

print("\nNew exact text matches:", (after_exact & ~before_exact).sum())

BEFORE CORRECTION
Exact matches: 16
Total: 2311
Exact match rate: 0.69%

AFTER CORRECTION
Exact matches: 58
Total: 2311
Exact match rate: 2.51%

New exact text matches: 42


In [213]:
display(
    prefix_df[
        [
            "doc_id",
            "label",
            "gold_text",
            "token_text",
            "suffix_extra",
            "gold_start",
            "gold_end",
            "token_start",
            "token_end",
            "end_diff"
        ]
    ].head(100)
)

,doc_id,label,gold_text,token_text,suffix_extra,gold_start,gold_end,token_start,token_end,end_diff
185,975074,COURT,Civil Court,(Civil Court,,629,640,628,640,0
216,975074,STAT,TP Act,(TP Act,,10756,10762,10755,10762,0
217,975074,STAT,BT&AL Act,(BT&AL Act,,11044,11053,11043,11053,0
1241,26855411,JUDGE,AJAY RASTOGI,(AJAY RASTOGI,,25129,25141,25128,25141,0
1303,62854486,JUDGE,MANOJ KUMAR OHRI,(MANOJ KUMAR OHRI,,18795,18811,18794,18811,0
1630,1161164,JUDGE,B. Sudershan Reddy,(B. Sudershan Reddy,,35211,35229,35210,35229,0
1698,95311661,JUDGE,ASHOK BHUSHAN,(ASHOK BHUSHAN,,17441,17454,17440,17454,0
1699,95311661,JUDGE,INDU MALHOTRA,(INDU MALHOTRA,,17514,17527,17513,17527,0
2404,1992267,JUDGE,ASOK KUMAR GANGULY,(ASOK KUMAR GANGULY,,21548,21566,21547,21566,0
2510,39503956,PREC,M.K Prasad vs. P. Arumugam,(M.K Prasad vs. P. Arumugam,,6889,6915,6888,6915,0


In [214]:
suffix_df = mismatches_df[
    mismatches_df["alignment_type"] == "SUFFIX_ONLY"
].copy()

suffix_counts = (
    suffix_df["suffix_extra"]
    .value_counts()
    .rename_axis("suffix")
    .reset_index(name="count")
)

suffix_counts["percentage"] = (
    100 * suffix_counts["count"] / len(suffix_df)
)

display(suffix_counts.head(50))

,suffix,count,percentage
0,",",1188,53.059402
1,.,931,41.581063
2,),38,1.697186
3,],16,0.714605
4,',14,0.625279
5,s,12,0.535954
6,al,9,0.401965
7,-,8,0.357302
8,'s,5,0.223314
9,"""",4,0.178651


In [215]:
display(
    suffix_df[
        [
            "doc_id",
            "label",
            "gold_text",
            "token_text",
            "suffix_extra",
            "gold_start",
            "gold_end",
            "token_start",
            "token_end",
            "end_diff"
        ]
    ].head(100)
)

,doc_id,label,gold_text,token_text,suffix_extra,gold_start,gold_end,token_start,token_end,end_diff
4,115651329,DATE,18.11.1994,"18.11.1994,",",",575,585,575,586,1
17,115651329,STAT,IPC,IPC.,.,2491,2494,2491,2495,1
19,115651329,STAT,IPC,IPC.,.,2685,2688,2685,2689,1
32,115651329,COURT,High Court of Punjab and Haryana,High Court of Punjab and Haryana.,.,4013,4045,4013,4046,1
38,115651329,COURT,High Court,High Court.,.,4425,4435,4425,4436,1
...,...,...,...,...,...,...,...,...,...,...
456,736324,DATE,from.1.9.1999,"from.1.9.1999,",",",10210,10223,10210,10224,1
462,736324,CASENO,O.A. No. 93/99,"O.A. No. 93/99,",",",11886,11900,11886,11901,1
464,736324,DATE,1.9.1999,"1.9.1999,",",",12046,12054,12046,12055,1
465,736324,PREC,Shankarsan Dash Vs. Union of India [1991 (3) S...,Shankarsan Dash Vs. Union of India [1991 (3) S...,.,12849,12901,12849,12902,1


In [216]:
suffix_by_label = (
    suffix_df
    .groupby(["label", "suffix_extra"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(suffix_by_label)

,label,suffix_extra,count
20,DATE,",",333
22,DATE,.,292
45,STAT,.,234
31,PREC,",",174
43,STAT,",",168
16,COURT,",",116
11,AUTH,.,98
14,CASENO,.,89
9,AUTH,",",84
17,COURT,.,80


In [217]:
interesting_suffixes = [
    "s",
    "al",
    "h",
    "6",
    "]",
    ")",
    "'s",
    ",J.",
]

interesting_df = suffix_df[
    suffix_df["suffix_extra"].isin(interesting_suffixes)
].copy()

display(
    interesting_df[
        [
            "doc_id",
            "label",
            "gold_text",
            "token_text",
            "suffix_extra",
            "gold_start",
            "gold_end",
            "token_start",
            "token_end"
        ]
    ].sort_values(
        ["suffix_extra", "label"]
    )
)

,doc_id,label,gold_text,token_text,suffix_extra,gold_start,gold_end,token_start,token_end
7864,1011232,AUTH,Forest Officer,Forest Officer's,'s,21709,21723,21709,21725
7899,1011232,AUTH,Compensation Officer,Compensation Officer's,'s,27543,27563,27543,27565
2913,19436407,COURT,Calcutta High Court,Calcutta High Court's,'s,5394,5413,5394,5415
7365,1404484,COURT,High Court,High Court's,'s,112081,112091,112081,112093
8137,661155,PREC,Ishwarsingh M. Rajput,Ishwarsingh M. Rajput's,'s,38083,38104,38083,38106
...,...,...,...,...,...,...,...,...,...
6545,1622258,COURT,High Court,High Courts,s,16147,16157,16147,16158
7235,1404484,COURT,High Court,High Courts,s,50612,50622,50612,50623
8034,156503,COURT,High Court,High Courts,s,32748,32758,32748,32759
8083,661155,COURT,High Court,High Courts,s,6924,6934,6924,6935


In [218]:
from collections import defaultdict, Counter

suffix_map_counts = defaultdict(Counter)

suffix_df = mismatches_df[
    mismatches_df["alignment_type"] == "SUFFIX_ONLY"
].copy()

for _, row in suffix_df.iterrows():

    key = (
        row["label"],
        row["token_text"]
    )

    suffix_map_counts[key][row["gold_text"]] += 1

In [219]:
rows = []

for (label, token_text), counter in suffix_map_counts.items():

    for gold_text, count in counter.items():

        rows.append({
            "label": label,
            "token_text": token_text,
            "gold_text": gold_text,
            "count": count
        })

suffix_map_df = pd.DataFrame(rows)

suffix_map_df = suffix_map_df.sort_values(
    "count",
    ascending=False
)

suffix_map_df.head(30)

,label,token_text,gold_text,count
3,COURT,High Court.,High Court,58
6,COURT,"High Court,",High Court,56
1,STAT,IPC.,IPC,31
380,STAT,Constitution of India.,Constitution of India,21
93,AUTH,State Government.,State Government,20
126,STAT,Constitution.,Constitution,20
7,STAT,"IPC,",IPC,17
1392,APP,Raunaq International Ltd.,Raunaq International Ltd,16
322,AUTH,Special Court.,Special Court,15
1291,STAT,"PMLA,",PMLA,14


In [220]:
ambiguity_df = (
    suffix_map_df
    .groupby(["label", "token_text"])
    .agg(
        num_gold_texts=("gold_text", "nunique"),
        total_count=("count", "sum")
    )
    .reset_index()
)

ambiguous = ambiguity_df[
    ambiguity_df["num_gold_texts"] > 1
].sort_values(
    "num_gold_texts",
    ascending=False
)

ambiguous.head(30)

,label,token_text,num_gold_texts,total_count


In [221]:
print("Total suffix patterns:", len(ambiguity_df))
print("Ambiguous patterns:", len(ambiguous))

print(
    "Unambiguous patterns:",
    (ambiguity_df["num_gold_texts"] == 1).sum()
)

Total suffix patterns: 1417
Ambiguous patterns: 0
Unambiguous patterns: 1417


In [222]:
suffix_map = {}

for (label, token_text), counter in suffix_map_counts.items():

    if len(counter) == 1:
        gold_text = next(iter(counter))
        suffix_map[(label, token_text)] = gold_text

print("Number of mappings:", len(suffix_map))

Number of mappings: 1417


In [223]:
for (label, token_text), gold_text in list(suffix_map.items())[:30]:
    print(
        f"{label:10s} | "
        f"{token_text!r:40s} -> {gold_text!r}"
    )

DATE       | '18.11.1994,'                            -> '18.11.1994'
STAT       | 'IPC.'                                   -> 'IPC'
COURT      | 'High Court of Punjab and Haryana.'      -> 'High Court of Punjab and Haryana'
COURT      | 'High Court.'                            -> 'High Court'
A.COUNSEL  | 'Vibha Datta Makhija,'                   -> 'Vibha Datta Makhija'
DATE       | '19.11.1994,'                            -> '19.11.1994'
COURT      | 'High Court,'                            -> 'High Court'
STAT       | 'IPC,'                                   -> 'IPC'
PREC       | 'State of Maharashtra v. Balram Bama Patil, (1983) 2 SCC 28,' -> 'State of Maharashtra v. Balram Bama Patil, (1983) 2 SCC 28'
PREC       | 'R. Prakash v. State of Karnataka (2004) 9 SCC 27.' -> 'R. Prakash v. State of Karnataka (2004) 9 SCC 27'
PREC       | 'Sevaka Perumal v. State of T.N.(1991) 3 SCC 471.' -> 'Sevaka Perumal v. State of T.N.(1991) 3 SCC 471'
DATE       | '14.10.2014,'                      

In [224]:
def evaluate_suffix_correction(suffix_df, suffix_map):
    results = []

    for _, row in suffix_df.iterrows():

        gold_start = row["gold_start"]
        gold_end = row["gold_end"]

        token_start = row["token_start"]
        token_end = row["token_end"]

        label = row["label"]
        token_text = row["token_text"]
        gold_text = row["gold_text"]

        # ---------------------------------------------------------
        # Pretend token span is the model prediction
        # ---------------------------------------------------------
        predicted_start = token_start
        predicted_end = token_end

        predicted_text = token_text

        # ---------------------------------------------------------
        # Apply learned suffix correction
        # ---------------------------------------------------------
        key = (label, predicted_text)

        if key in suffix_map:

            corrected_text = suffix_map[key]

            # SUFFIX_ONLY means the start is already correct.
            corrected_start = predicted_start
            corrected_end = (
                corrected_start + len(corrected_text)
            )

            corrected = True

        else:

            corrected_text = predicted_text
            corrected_start = predicted_start
            corrected_end = predicted_end

            corrected = False

        # ---------------------------------------------------------
        # Compare corrected span with gold
        # ---------------------------------------------------------
        exact_before = (
            predicted_start == gold_start
            and predicted_end == gold_end
        )

        exact_after = (
            corrected_start == gold_start
            and corrected_end == gold_end
        )

        results.append({
            "label": label,
            "gold_text": gold_text,

            "gold_start": gold_start,
            "gold_end": gold_end,

            "predicted_text": predicted_text,
            "predicted_start": predicted_start,
            "predicted_end": predicted_end,

            "corrected_text": corrected_text,
            "corrected_start": corrected_start,
            "corrected_end": corrected_end,

            "corrected": corrected,

            "exact_before": exact_before,
            "exact_after": exact_after
        })

    results_df = pd.DataFrame(results)

    return results_df


In [225]:
suffix_eval_df = evaluate_suffix_correction(
    suffix_df,
    suffix_map
)

In [226]:
print(
    "Exact before:",
    suffix_eval_df["exact_before"].sum()
)

print(
    "Exact after:",
    suffix_eval_df["exact_after"].sum()
)

print(
    "Total:",
    len(suffix_eval_df)
)

Exact before: 0
Exact after: 2239
Total: 2239


In [227]:
suffix_map = {}

for (label, token_text), counter in suffix_map_counts.items():
    if len(counter) == 1:
        suffix_map[(label, token_text)] = next(iter(counter))

In [228]:
# def find_partial_token_overlaps(
#     train_text,
#     train_gold,
#     tokenizer
# ):
#     partial = []

#     for doc_id, text in train_text.items():

#         token_ids, offsets = encode_with_offsets(text)

#         for ann in train_gold[doc_id]:

#             gold_start, gold_end, label, gold_text = ann

#             for token_idx, (tok_start, tok_end) in enumerate(offsets):

#                 overlaps = (
#                     tok_start < gold_end
#                     and tok_end > gold_start
#                 )

#                 if not overlaps:
#                     continue

#                 # Token crosses the gold START
#                 crosses_start = (
#                     tok_start < gold_start < tok_end
#                 )

#                 # Token crosses the gold END
#                 crosses_end = (
#                     tok_start < gold_end < tok_end
#                 )

#                 if crosses_start or crosses_end:

#                     partial.append({
#                         "doc_id": doc_id,
#                         "gold": ann,
#                         "token_idx": token_idx,
#                         "token_id": token_ids[token_idx],
#                         "token_offset": (tok_start, tok_end),
#                         "token_text": text[tok_start:tok_end],
#                         "crosses_start": crosses_start,
#                         "crosses_end": crosses_end
#                     })

#     return partial
# partial = find_partial_token_overlaps(
#     train_text,
#     train_gold,
#     tokenizer
# )

# print("Partial token/entity overlaps:", len(partial))
# result =[]
# for x in partial[:30]:
#     result.append(x["token_text"])

In [229]:
# maxi = 4 
# for x in partial:
#    maxi = min(maxi, x["token_offset"][1]-x["gold"][1])
# print(maxi)

In [230]:
# for x in partial:
#     print(
#         "GOLD:",
#         x["gold"],
#         "| TOKEN:",
#         repr(x["token_text"]),
#         "| TOKEN OFFSET:",
#         x["token_offset"]
#     )

In [231]:
# unique_failing_chars = []
# for x in partial:
#     unique_failing_chars.append(x["token_text"])
# failing_set = set(unique_failing_chars)
# print(failing_set)
# print("Total failing chars",len(failing_set))

In [232]:
def align_tokens_to_gold(text, token_ids, offsets, gold_annotations):
    """
    Convert character-level gold annotations into token-level BIO labels.

    Parameters
    ----------
    text : str
        Original document text.

    token_ids : list[int]
        BPE token IDs.

    offsets : list[tuple[int, int]]
        Character offsets for each BPE token.
        Each offset is [start, end), Python character indexing.

    gold_annotations : list
        Each annotation:
        [start, end, label, entity_text]

    Returns
    -------
    token_labels : list[str]
        BIO labels for every token.

    token_entity_spans : list[tuple[int, int] | None]
        Exact character span contributed by the entity for each token.
    """

    token_labels = ["O"] * len(token_ids)
    token_entity_spans = [None] * len(token_ids)

    for gold_start, gold_end, gold_label, gold_text in gold_annotations:

        matched_tokens = []

        for token_idx, (tok_start, tok_end) in enumerate(offsets):

            # Character intersection
            overlap_start = max(tok_start, gold_start)
            overlap_end = min(tok_end, gold_end)

            if overlap_start < overlap_end:
                matched_tokens.append(
                    (token_idx, overlap_start, overlap_end)
                )

        if not matched_tokens:
            print(
                f"WARNING: no token matched "
                f"{gold_label}: {gold_text!r} "
                f"[{gold_start}, {gold_end})"
            )
            continue

        # Sort by token position
        matched_tokens.sort(key=lambda x: x[0])

        # First token = B
        token_idx, overlap_start, overlap_end = matched_tokens[0]

        token_labels[token_idx] = f"B-{gold_label}"
        token_entity_spans[token_idx] = (
            overlap_start,
            overlap_end
        )

        # Remaining tokens = I
        for token_idx, overlap_start, overlap_end in matched_tokens[1:]:

            token_labels[token_idx] = f"I-{gold_label}"
            token_entity_spans[token_idx] = (
                overlap_start,
                overlap_end
            )

    return token_labels, token_entity_spans

In [233]:
text = train_text["115651329"]
gold = train_gold["115651329"]

token_ids, offsets = encode_with_offsets(text)

token_labels, token_entity_spans = align_tokens_to_gold(
    text=text,
    token_ids=token_ids,
    offsets=offsets,
    gold_annotations=gold
)

In [235]:
LABELS = [
    "O",
    "B-APP", "I-APP",
    "B-RESP", "I-RESP",
    "B-A.COUNSEL", "I-A.COUNSEL",
    "B-R.COUNSEL", "I-R.COUNSEL",
    "B-JUDGE", "I-JUDGE",
    "B-WIT", "I-WIT",
    "B-AUTH", "I-AUTH",
    "B-COURT", "I-COURT",
    "B-STAT", "I-STAT",
    "B-PREC", "I-PREC",
    "B-DATE", "I-DATE",
    "B-CASENO", "I-CASENO",
]

label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

NUM_LABELS = len(LABELS)

print(NUM_LABELS)

25


In [236]:
def prepare_document(doc_id, train_text, train_gold, tokenizer):
    text = train_text[doc_id]
    gold_annotations = train_gold[doc_id]

    token_ids, offsets = encode_with_offsets(text)

    token_labels, entity_spans = align_tokens_to_gold(
        text=text,
        token_ids=token_ids,
        offsets=offsets,
        gold_annotations=gold_annotations
    )

    label_ids = [
        label2id[label]
        for label in token_labels
    ]

    return {
        "doc_id": doc_id,
        "input_ids": token_ids,
        "labels": label_ids,
        "offsets": offsets,
        "entity spans" : entity_spans
    }

In [237]:
import random

doc_ids = list(train_text.keys())

rng = random.Random(SEED)
rng.shuffle(doc_ids)

split = int(0.9 * len(doc_ids))

train_doc_ids = doc_ids[:split]
val_doc_ids = doc_ids[split:]

print("Train documents:", len(train_doc_ids))
print("Val documents:", len(val_doc_ids))

Train documents: 63
Val documents: 7


In [238]:
def chunk_document(example, max_len=1024):
    input_ids = example["input_ids"]
    labels = example["labels"]
    offsets = example["offsets"]

    chunks = []

    for start in range(0, len(input_ids), max_len):
        end = min(start + max_len, len(input_ids))

        chunks.append({
            "doc_id": example["doc_id"],
            "input_ids": input_ids[start:end],
            "labels": labels[start:end],
            "offsets": offsets[start:end],
        })

    return chunks

In [239]:
train_chunks = []
val_chunks = []

for doc_id in train_doc_ids:

    example = prepare_document(
        doc_id,
        train_text,
        train_gold,
        converter
    )

    train_chunks.extend(
        chunk_document(example, max_len=1024)
    )


for doc_id in val_doc_ids:

    example = prepare_document(
        doc_id,
        train_text,
        train_gold,
        converter
    )

    val_chunks.extend(
        chunk_document(example, max_len=1024)
    )

print("Train chunks:", len(train_chunks))
print("Val chunks:", len(val_chunks))

Train chunks: 455
Val chunks: 64


In [240]:
# from collections import Counter

# counts = Counter()

# for example in bio_dataset:
#     counts.update(example["labels"])

# for label_id, count in counts.most_common():
#     print(f"{id2label[label_id]:15s} {count:10d}")

In [241]:
import torch
from torch.utils.data import Dataset, DataLoader


class BIODataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        item = self.chunks[idx]

        return {
            "input_ids": torch.tensor(
                item["input_ids"],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                item["labels"],
                dtype=torch.long
            )
        }
def bio_collate_fn(batch, pad_id):

    max_len = max(
        item["input_ids"].size(0)
        for item in batch
    )

    batch_size = len(batch)

    input_ids = torch.full(
        (batch_size, max_len),
        pad_id,
        dtype=torch.long
    )

    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        length = item["input_ids"].size(0)

        input_ids[i, :length] = item["input_ids"]
        labels[i, :length] = item["labels"]
        attention_mask[i, :length] = 1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

In [242]:
PAD_ID = 0  # if your tokenizer has this

train_dataset = BIODataset(train_chunks)
val_dataset = BIODataset(val_chunks)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=lambda batch: bio_collate_fn(batch, PAD_ID),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=lambda batch: bio_collate_fn(batch, PAD_ID),
)

In [243]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)
print(batch["attention_mask"].shape)

torch.Size([8, 1024])
torch.Size([8, 1024])
torch.Size([8, 1024])


In [244]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [245]:

# TODO 3.2: integrate RoPE through attention and the decoder architecture in this section.
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor,attn_mask=None) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim = 1)
            v = v.repeat_interleave(self.n_rep, dim = 1)
       
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.attn_dropout.p if self.training else 0.0
        )
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out



In [246]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook



def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE


def build_attn_mask(is_causal, padding_mask, T):
    """is_causal (B,) bool; padding_mask (B,T) bool, True = padding, or None -> (B,1,T,T) bool."""
    full = torch.ones(T, T, dtype=torch.bool, device=is_causal.device)
    allow = torch.where(is_causal[:, None, None, None], torch.tril(full), full)
    if padding_mask is not None:
        allow = allow & ~padding_mask[:, None, None, :].to(torch.bool)
    return allow
    
class EncoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = EncoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.ffn(self.ln2(x))
        return x   # unchanged from Question 1


class EncoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )
        
        self.lm_head.weight = self.tok_emb.weight
                

    def forward(self, idx, padding_mask=None, is_causal=None):                 # CHANGED: is_causal
            B, T = idx.shape
            if padding_mask is not None:
                if padding_mask.shape != (B, T):
                    raise ValueError("padding_mask must have shape (B, T)")
                padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
            if is_causal is None:                                                  # old behaviour: bidirectional
                is_causal = torch.zeros(B, dtype=torch.bool, device=idx.device)
            attn_mask = build_attn_mask(is_causal, padding_mask, T)
            x = self.tok_emb(idx)
            if self.cfg.pos_emb_type == "sinusoidal":
                x = x + self.pos_emb[:T].unsqueeze(0)
            for block in self.blocks:
                x = block(x, attn_mask=attn_mask)
            x = self.ln_f(x)
            return self.lm_head(x) 


In [247]:
class LoraLinear(nn.Module):
    def __init__(self, layer, rank=8, alpha=16, dropout=0.05):
        super().__init__()

        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout)

        # Freeze original layer
        for p in self.layer.parameters():
            p.requires_grad = False

        self.in_features = layer.in_features
        self.out_features = layer.out_features

        # A: [rank, in_features]
        self.A = nn.Parameter(
            torch.empty(rank, self.in_features)
        )

        # B: [out_features, rank]
        self.B = nn.Parameter(
            torch.zeros(self.out_features, rank)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=5**0.5
        )

    def forward(self, x):

        # Original path: W0 x
        W0x = self.layer(x)

        # LoRA path: BAx
        Ax = F.linear(self.dropout(x), self.A)
        BAx = F.linear( Ax, self.B)

        return W0x + (self.alpha / self.rank) * BAx

In [248]:
class LoraModel(nn.Module):
    def __init__(self, model, d_model, rank = 8, alpha =16, dropout =0.05, labels =25, projections = ("q_proj","k_proj")):
        super().__init__()
        self.model = model
        self.d_model = d_model
        self.projections = tuple(projections)
        # Freeze Phase-1 model
        for param in self.model.parameters():
            param.requires_grad = False
        self._add_lora(rank, alpha, dropout)
        self.ner_head = nn.Linear(d_model, labels)

    def _add_lora(self, rank, alpha, dropout):
        for module in self.model.modules():
            for proj_name in self.projections:
    
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2
    
                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoraLinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            )
    
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)
    
                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoraLinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            ),
                        )
    def  get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        # Token embeddings
        x = self.model.tok_emb(idx)
        # Sinusoidal positional encoding if Phase-1 used it
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
        # Build attention mask
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool
            )
        # Encoder is bidirectional
        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device
        )
        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T
        )
        # Transformer blocks
        for block in self.model.blocks:
            x = block(
                x,
                attn_mask=attn_mask
            )
        x = self.model.ln_f(x)

        return x
    def forward(self, x, padding_mask=None):
          hidden_states = self.get_hidden_states(x, padding_mask)
          logits = self.ner_head(hidden_states)
          return logits


In [249]:
class LoraPartialFinetuneModel(nn.Module):
    def __init__(self, model, d_model, rank = 8, alpha =16, dropout =0.05, labels =25, projections = ("q_proj","k_proj"), unfreeze_last_n = 0):
        super().__init__()
        self.model = model
        self.d_model = d_model
        self.projections = tuple(projections)
        # Freeze Phase-1 model
        for param in self.model.parameters():
            param.requires_grad = False
        if unfreeze_last_n > 0:
            if unfreeze_last_n > len(self.model.blocks):
                raise ValueError("unfreeze_last_n exceeds number of blocks")

            for block in self.model.blocks[-unfreeze_last_n:]:
                for param in block.parameters():
                    param.requires_grad = True
        self._add_lora(rank, alpha, dropout)
        self.ner_head = nn.Linear(d_model, labels)

    def _add_lora(self, rank, alpha, dropout):
        for module in self.model.modules():
            for proj_name in self.projections:
    
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2
    
                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoraLinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            )
    
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)
    
                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoraLinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            ),
                        )
    def  get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        # Token embeddings
        x = self.model.tok_emb(idx)
        # Sinusoidal positional encoding if Phase-1 used it
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
        # Build attention mask
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool
            )
        # Encoder is bidirectional
        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device
        )
        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T
        )
        # Transformer blocks
        for block in self.model.blocks:
            x = block(
                x,
                attn_mask=attn_mask
            )
        x = self.model.ln_f(x)

        return x
    def forward(self, x, padding_mask=None):
          hidden_states = self.get_hidden_states(x, padding_mask)
          logits = self.ner_head(hidden_states)
          return logits


In [250]:
class LoRMALinear(nn.Module):
    def __init__(self, layer, rank=16, alpha=32, dropout=0.05, variant = "plus"):
        super().__init__()
        if variant not in {"plus", "pi"}:
            raise ValueError("Only phi or plus supported")
        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        self.variant = variant
        self.dropout = nn.Dropout(dropout)

        for p in self.layer.parameters():
            p.requires_grad = False
        d = layer.out_features
        self.A = nn.Parameter(torch.empty(rank, d))
        self.B = nn.Parameter(torch.empty(d, rank))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=5**0.5)
            nn.init.zeros_(self.B)
        else:
            nn.init.normal_(self.B)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A.zero_()
                self.A[0, 0] = 1.0
    @staticmethod
    def inflate_pi(BA):
        """Cyclically rotate row i by i positions."""
        d = BA.shape[0]
        return torch.stack(
            [torch.roll(BA[i], shifts=i, dims=0) for i in range(d)],
            dim=0,
        )
    def forward(self, x):
        base = self.layer(x)

        # Apply BA to the pretrained output
        z = self.dropout(base)
        if self.variant == "plus":
            Az = F.linear(z, self.A)
            BAz = F.linear(Az, self.B)
    
            # h = (I + alpha/rank * BA) W0 x
            output = base + (self.alpha / self.rank) * BAz
            
    
            return output
        else:
            BA = self.B @ self.A

            # Permutation-based rank inflation
            BA_pi = self.inflate_pi(BA)
    
            # h = (I_pi(BA) @ W0) x
            # Equivalent to applying BA_pi to W0 x
            adapted = F.linear(z, BA_pi)
    
            return base + (self.alpha / self.rank) * adapted

In [251]:
class LormaModel(nn.Module):
    def __init__(self, model, d_model, rank = 8, alpha =16, dropout =0.05, labels =25, variant="plus", projections=("q_proj", "k_proj")):
        super().__init__()
        self.model = model
        self.d_model = d_model
        self.projections = tuple(projections)
        self.variant = variant
        # Freeze Phase-1 model
        for param in self.model.parameters():
            param.requires_grad = False
        self._add_lorma(rank, alpha, dropout)
        self.ner_head = nn.Linear(d_model, labels)


    def _add_lorma(self, rank, alpha, dropout):
        for module in self.model.modules():
            for proj_name in self.projections:
    
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2
    
                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoRMALinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                                variant = self.variant
                            )
    
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)
    
                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoRMALinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                                variant = self.variant
                            ),
                        )
    def  get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        # Token embeddings
        x = self.model.tok_emb(idx)
        # Sinusoidal positional encoding if Phase-1 used it
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
        # Build attention mask
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool
            )
        # Encoder is bidirectional
        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device
        )
        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T
        )
        # Transformer blocks
        for block in self.model.blocks:
            x = block(
                x,
                attn_mask=attn_mask
            )
        x = self.model.ln_f(x)

        return x
    def forward(self, x, padding_mask=None):
          hidden_states = self.get_hidden_states(x, padding_mask)
          logits = self.ner_head(hidden_states)
          return logits


In [252]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=10,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model_phase1 = EncoderLM(cfg)

model_phase1 = model_phase1.to(device)

In [253]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model_phase1))

total parameters :  56592384


In [254]:
checkpoint = torch.load(
    "/kaggle/input/datasets/rogue2222/hybrid1b/hybrid_encoder.pt",
    map_location=device,
    weights_only=False
)
state_dict = checkpoint["model_state_dict"]

new_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    new_state_dict[key] = value

model_phase1.load_state_dict(new_state_dict)

<All keys matched successfully>

In [255]:
class NERHead(nn.Module):
    def __init__(self, model, d_model, num_classes):
        super().__init__()
        self.model = model
        self.head = nn.Linear(d_model, num_classes)
    
    def  get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        # Token embeddings
        x = self.model.tok_emb(idx)
        # Sinusoidal positional encoding if Phase-1 used it
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T].unsqueeze(0)
        # Build attention mask
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError(
                    "padding_mask must have shape (B, T)"
                )
            padding_mask = padding_mask.to(
                device=idx.device,
                dtype=torch.bool
            )
        # Encoder is bidirectional
        is_causal = torch.zeros(
            B,
            dtype=torch.bool,
            device=idx.device
        )
        attn_mask = build_attn_mask(
            is_causal,
            padding_mask,
            T
        )
        for block in self.model.blocks:
                x = block(
                    x,
                    attn_mask=attn_mask
                )
        x = self.model.ln_f(x)
        return x
    def forward(self, input_ids, padding_mask=None):
        hidden = self.get_hidden_states(
            input_ids,
            padding_mask=padding_mask
        )
        logits = self.head(hidden)
        return logits

In [256]:
NUM_LABELS = 25

# model = LormaModel(
#     model=model_phase1,
#     d_model=MODEL_DIM,
#     rank=16,
#     alpha=32,
#     dropout=0.05,
#     labels=NUM_LABELS,
#     # ("q_proj", "k_proj", "v_proj", "o_proj", "net0", "net1")
#     projections =("q_proj", "v_proj","net0", "net1"),
#     # unfreeze_last_n = 1,
#     variant = "pi"
    
    
# )
model = NERHead(model_phase1, d_model =MODEL_DIM, num_classes = 25)
model =model.to(device)

In [257]:
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)
model = model.to(device)

Using 2 GPUs


In [258]:
import torch
from sklearn.metrics import f1_score


@torch.no_grad()
def evaluate_token_f1(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    for batch in loader:

        input_ids = batch["input_ids"].to(
            device
        )

        labels = batch["labels"].to(
            device
        )

        attention_mask = batch["attention_mask"].to(
            device)

        # True = padding, False = real token
        padding_mask = ~attention_mask.bool()

        logits = model(
            input_ids,
            padding_mask=padding_mask
        )

        preds = logits.argmax(dim=-1)

        # Ignore padded positions
        valid_mask = labels != -100

        all_preds.extend(
            preds[valid_mask].cpu().tolist()
        )

        all_labels.extend(
            labels[valid_mask].cpu().tolist()
        )

    micro_f1 = f1_score(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return micro_f1, macro_f1

In [259]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=20,
    patience=3,
    min_delta=1e-4,
    max_grad_norm=1.0,
    use_amp=True,
    save_path="best_hybrid_ner.pt"
):
    """
    Train Hybrid + NER with:
      - padding-aware attention
      - AMP/bfloat16
      - gradient clipping
      - ReduceLROnPlateau
      - early stopping
      - best-checkpoint saving

    Checkpoint / early stopping metric:
        Validation Macro-F1

    Also reports:
        Validation Micro-F1
    """

    model.to(device)

    # ------------------------------------------------------------
    # LR scheduler
    # ------------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    # ------------------------------------------------------------
    # AMP
    # ------------------------------------------------------------

    amp_enabled = (
        use_amp and
        device.type == "cuda"
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled
    )

    # ------------------------------------------------------------
    # Best model tracking
    # ------------------------------------------------------------

    best_macro_f1 = -float("inf")
    best_micro_f1 = -float("inf")

    epochs_without_improvement = 0

    history = []

    # ============================================================
    # EPOCH LOOP
    # ============================================================

    for epoch in range(1, epochs + 1):

        # ========================================================
        # TRAIN
        # ========================================================

        model.train()

        total_loss = 0.0
        total_tokens = 0

        for batch in train_loader:

            input_ids = batch["input_ids"].to(
                device,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                non_blocking=True
            )

            attention_mask = batch["attention_mask"].to(
                device,
                non_blocking=True
            )

            # True  = padding
            # False = real token
            padding_mask = ~attention_mask.bool()

            optimizer.zero_grad(set_to_none=True)

            # ----------------------------------------------------
            # Forward
            # ----------------------------------------------------

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled
            ):

                logits = model(
                    input_ids,
                    padding_mask=padding_mask
                )

                loss = criterion(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1)
                )

            # ----------------------------------------------------
            # Backward
            # ----------------------------------------------------

            scaler.scale(loss).backward()

            # Unscale before gradient clipping
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_grad_norm
            )

            scaler.step(optimizer)
            scaler.update()

            # ----------------------------------------------------
            # Loss accumulation
            # ----------------------------------------------------

            valid_tokens = (
                labels != -100
            ).sum().item()

            total_loss += (
                loss.item() * valid_tokens
            )

            total_tokens += valid_tokens

        train_loss = (
            total_loss / max(total_tokens, 1)
        )

        # ========================================================
        # VALIDATION
        # ========================================================

        val_micro_f1, val_macro_f1 = evaluate_token_f1(
            model=model,
            loader=val_loader,
            device=device
        )

        # Scheduler monitors Macro-F1
        scheduler.step(val_macro_f1)

        current_lr = optimizer.param_groups[0]["lr"]

        # ========================================================
        # LOG
        # ========================================================

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Micro-F1: {val_micro_f1:.4f} | "
            f"Val Macro-F1: {val_macro_f1:.4f} | "
            f"LR: {current_lr:.2e}"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_micro_f1": val_micro_f1,
            "val_macro_f1": val_macro_f1,
            "lr": current_lr
        })

        # ========================================================
        # BEST CHECKPOINT
        # ========================================================

        if val_macro_f1 > best_macro_f1 + min_delta:

            best_macro_f1 = val_macro_f1
            best_micro_f1 = val_micro_f1

            epochs_without_improvement = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_macro_f1": best_macro_f1,
                    "best_micro_f1": best_micro_f1,
                },
                save_path
            )

            print(
                f"  ✓ Saved best model | "
                
                f"Macro-F1: {best_macro_f1:.4f} | "
                f"Micro-F1: {best_micro_f1:.4f}"
            )

        else:

            epochs_without_improvement += 1

            print(
                f"  No Macro-F1 improvement "
                f"({epochs_without_improvement}/{patience})"
            )

            if epochs_without_improvement >= patience:

                print(
                    f"\nEarly stopping at epoch {epoch}."
                )

                break

    # ============================================================
    # FINAL
    # ============================================================
    checkpoint = torch.load(
    save_path,
    map_location=device,
    weights_only=True
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    
    print(f"Restored best model from epoch {checkpoint['epoch']}")

    print("\nTraining complete.")
    print(
        f"Best Val Macro-F1 : {best_macro_f1:.4f}"
    )
    print(
        f"Best Val Micro-F1 : {best_micro_f1:.4f}"
    )

    return history

In [260]:
def distillation_loss(student_logits, teacher_logits, labels, criterion, alpha =0.5, temperature = 2.0):
    # 
    ce_loss = criterion(
        student_logits.reshape(-1, student_logits.size(-1)),
        labels.reshape(-1),
    )
    num_labels = student_logits.size(-1)
    student_log_probs = F.log_softmax(student_logits/temperature, dim =-1)
    teacher_probs = F.softmax(teacher_logits/ temperature, dim =-1)
    kl_per_token = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none",
    ).sum(dim=-1)
    valid_mask = labels != -100
    if valid_mask.any():
        kl_loss = kl_per_token[valid_mask].mean()
    else:
        kl_loss = student_logits.sum() * 0.0
    loss = ((1.0 - alpha) * ce_loss + alpha*(temperature**2)*kl_loss)

    return loss, ce_loss.detach(), kl_loss.detach()
    

In [261]:
import torch
import torch.nn.functional as F


def train_model_distillation(
    student,
    teacher,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs=20,
    patience=3,
    min_delta=1e-4,
    max_grad_norm=1.0,
    use_amp=True,
    temperature=2.0,
    alpha=0.5,
    save_path="best_distilled_ner.pt",
):
    """
    Train a student NER model using knowledge distillation.

    Features:
      - Frozen teacher model
      - Gold-label cross-entropy + teacher KL divergence
      - Padding-aware attention
      - AMP
      - Gradient clipping
      - ReduceLROnPlateau
      - Early stopping on validation Macro-F1
      - Best-checkpoint saving and restoration

    Assumes evaluate_token_f1(model, loader, device)
    returns (micro_f1, macro_f1).
    """

    student.to(device)
    teacher.to(device)

    # ------------------------------------------------------------
    # Freeze teacher
    # ------------------------------------------------------------

    teacher.eval()

    for param in teacher.parameters():
        param.requires_grad = False

    # ------------------------------------------------------------
    # Scheduler
    # ------------------------------------------------------------

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5,
    )

    # ------------------------------------------------------------
    # AMP
    # ------------------------------------------------------------

    amp_enabled = use_amp and device.type == "cuda"

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled,
    )

    # ------------------------------------------------------------
    # Best-model tracking
    # ------------------------------------------------------------

    best_macro_f1 = -float("inf")
    best_micro_f1 = -float("inf")
    epochs_without_improvement = 0
    history = []

    # ============================================================
    # EPOCH LOOP
    # ============================================================

    for epoch in range(1, epochs + 1):

        student.train()
        teacher.eval()

        total_loss = 0.0
        total_ce_loss = 0.0
        total_kl_loss = 0.0
        total_tokens = 0

        # --------------------------------------------------------
        # TRAIN
        # --------------------------------------------------------

        for batch in train_loader:

            input_ids = batch["input_ids"].to(
                device, non_blocking=True
            )
            labels = batch["labels"].to(
                device, non_blocking=True
            )
            attention_mask = batch["attention_mask"].to(
                device, non_blocking=True
            )

            padding_mask = ~attention_mask.bool()

            optimizer.zero_grad(set_to_none=True)

            # Teacher predictions: no gradients
            with torch.no_grad():
                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=amp_enabled,
                ):
                    teacher_logits = teacher(
                        input_ids,
                        padding_mask=padding_mask,
                    )

            # Student predictions
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                student_logits = student(
                    input_ids,
                    padding_mask=padding_mask,
                )

                loss, ce_loss, kl_loss = distillation_loss(
                    student_logits=student_logits,
                    teacher_logits=teacher_logits,
                    labels=labels,
                    temperature=temperature,
                    alpha=alpha,
                    criterion = criterion
                )

            # Backward
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            trainable_params = [
                p for p in student.parameters()
                if p.requires_grad
            ]

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_grad_norm,
            )

            scaler.step(optimizer)
            scaler.update()

            # Accumulate losses
            valid_tokens = (labels != -100).sum().item()

            total_loss += loss.detach().item() * valid_tokens
            total_ce_loss += ce_loss.item() * valid_tokens
            total_kl_loss += kl_loss.item() * valid_tokens
            total_tokens += valid_tokens

        train_loss = total_loss / max(total_tokens, 1)
        avg_ce_loss = total_ce_loss / max(total_tokens, 1)
        avg_kl_loss = total_kl_loss / max(total_tokens, 1)

        # --------------------------------------------------------
        # VALIDATION
        # --------------------------------------------------------

        val_micro_f1, val_macro_f1 = evaluate_token_f1(
            model=student,
            loader=val_loader,
            device=device,
        )

        scheduler.step(val_macro_f1)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:02d} | "
            f"Loss: {train_loss:.4f} | "
            f"CE: {avg_ce_loss:.4f} | "
            f"KL: {avg_kl_loss:.4f} | "
            f"Val Micro-F1: {val_micro_f1:.4f} | "
            f"Val Macro-F1: {val_macro_f1:.4f} | "
            f"LR: {current_lr:.2e}"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "ce_loss": avg_ce_loss,
            "kl_loss": avg_kl_loss,
            "val_micro_f1": val_micro_f1,
            "val_macro_f1": val_macro_f1,
            "lr": current_lr,
        })

        # --------------------------------------------------------
        # SAVE BEST CHECKPOINT
        # --------------------------------------------------------

        if val_macro_f1 > best_macro_f1 + min_delta:

            best_macro_f1 = val_macro_f1
            best_micro_f1 = val_micro_f1
            epochs_without_improvement = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": student.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_macro_f1": best_macro_f1,
                    "best_micro_f1": best_micro_f1,
                },
                save_path,
            )

            print(
                f"  Saved best student | "
                f"Macro-F1: {best_macro_f1:.4f} | "
                f"Micro-F1: {best_micro_f1:.4f}"
            )

        else:
            epochs_without_improvement += 1

            print(
                f"  No improvement "
                f"({epochs_without_improvement}/{patience})"
            )

            if epochs_without_improvement >= patience:
                print(f"\nEarly stopping at epoch {epoch}.")
                break

    # ============================================================
    # RESTORE BEST STUDENT
    # ============================================================

    checkpoint = torch.load(
        save_path,
        map_location=device,
        weights_only=True,
    )

    student.load_state_dict(checkpoint["model_state_dict"])

    print("\nDistillation training complete.")
    print(f"Restored best epoch: {checkpoint['epoch']}")
    print(f"Best Val Macro-F1: {checkpoint['best_macro_f1']:.4f}")
    print(f"Best Val Micro-F1: {checkpoint['best_micro_f1']:.4f}")

    return history

In [262]:
# trainable_params = [
#     p for p in model.parameters()
#     if p.requires_grad
# ]


In [263]:
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable:", trainable)
print("Total:", total)
print("Trainable %:", 100 * trainable / total)

Trainable: 56605209
Total: 56605209
Trainable %: 100.0


In [264]:
# counts = torch.zeros(NUM_LABELS)

# for example in train_chunks:
#     for label_id in example["labels"]:
#         if label_id != -100:
#             counts[label_id] += 1

# total = counts.sum()

# # Negative examples for each class
# negative = total - counts

# # sqrt(negative / positive)
# weights = torch.sqrt(negative / counts)

# # Normalize average weight to 1
# weights = weights / weights.mean()

# print("Class weights:")
# for i in range(NUM_LABELS):
#     print(f"{id2label[i]:15s} {weights[i].item():.4f}")

In [265]:
counts = torch.zeros(NUM_LABELS)

for example in train_chunks:
    for label_id in example["labels"]:
        if label_id != -100:
            counts[label_id] += 1

# total = counts.sum()

# # Negative examples for each class
# negative = total - counts

# # sqrt(negative / positive)
# weights = torch.sqrt(negative / counts)

# # Normalize average weight to 1
# weights = weights / weights.mean()

# print("Class weights:")
# for i in range(NUM_LABELS):
#     print(f"{id2label[i]:15s} {weights[i].item():.4f}")
ent_types = ["APP","RESP","A.COUNSEL","R.COUNSEL","JUDGE","WIT",
             "AUTH","COURT","STAT","PREC","DATE","CASENO"]

cls_count = {t: counts[label2id[f"B-{t}"]] + counts[label2id[f"I-{t}"]] for t in ent_types}
cls_w = {t: (total / cls_count[t]) ** 0.25 for t in ent_types}     # mild power

weights = torch.ones(NUM_LABELS)
weights[label2id["O"]] = 0.3
for t in ent_types:
    weights[label2id[f"B-{t}"]] = cls_w[t] * 1.5     # fixed B:I ratio
    weights[label2id[f"I-{t}"]] = cls_w[t] * 1.0
weights[1:] = weights[1:] / weights[1:].mean()

In [266]:
print("O weight:", weights[label2id["O"]].item())

for t in ent_types:
    print(
        f"{t:12s} "
        f"B={weights[label2id[f'B-{t}']].item():.4f} "
        f"I={weights[label2id[f'I-{t}']].item():.4f}"
    )

O weight: 0.30000001192092896
APP          B=1.2283 I=0.8189
RESP         B=1.2404 I=0.8269
A.COUNSEL    B=1.5206 I=1.0137
R.COUNSEL    B=1.5395 I=1.0264
JUDGE        B=1.2964 I=0.8643
WIT          B=1.5639 I=1.0426
AUTH         B=1.1643 I=0.7762
COURT        B=1.1610 I=0.7740
STAT         B=1.0060 I=0.6707
PREC         B=0.7724 I=0.5149
DATE         B=0.9555 I=0.6370
CASENO       B=0.9517 I=0.6345


In [267]:
criterion = nn.CrossEntropyLoss(
    weight=weights.to(device),
    ignore_index=-100
)
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-5,
    weight_decay=0.01
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=30,
    patience=5,
    save_path="best_hybrid_ner.pt"
)

Epoch 01 | Train Loss: 1.9435 | Val Micro-F1: 0.9000 | Val Macro-F1: 0.0379 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.0379 | Micro-F1: 0.9000
Epoch 02 | Train Loss: 0.9355 | Val Micro-F1: 0.9087 | Val Macro-F1: 0.0560 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.0560 | Micro-F1: 0.9087
Epoch 03 | Train Loss: 0.7022 | Val Micro-F1: 0.9319 | Val Macro-F1: 0.1809 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.1809 | Micro-F1: 0.9319
Epoch 04 | Train Loss: 0.5532 | Val Micro-F1: 0.9461 | Val Macro-F1: 0.2606 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.2606 | Micro-F1: 0.9461
Epoch 05 | Train Loss: 0.4616 | Val Micro-F1: 0.9528 | Val Macro-F1: 0.3446 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.3446 | Micro-F1: 0.9528
Epoch 06 | Train Loss: 0.3992 | Val Micro-F1: 0.9547 | Val Macro-F1: 0.3824 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.3824 | Micro-F1: 0.9547
Epoch 07 | Train Loss: 0.3505 | Val Micro-F1: 0.9570 | Val Macro-F1: 0.4262 | LR: 1.00e-05
  ✓ Saved

In [268]:
# criterion = nn.CrossEntropyLoss(
#     weight=weights.to(device),
#     ignore_index=-100
# )
# trainable_params = [
#     p for p in model.parameters()
#     if p.requires_grad
# ]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-5,
    weight_decay=0.01
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=15,
    patience=5,
    save_path="best_hybrid_ner.pt"
)

Epoch 01 | Train Loss: 0.0867 | Val Micro-F1: 0.9677 | Val Macro-F1: 0.6287 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.6287 | Micro-F1: 0.9677
Epoch 02 | Train Loss: 0.0801 | Val Micro-F1: 0.9687 | Val Macro-F1: 0.6513 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.6513 | Micro-F1: 0.9687
Epoch 03 | Train Loss: 0.0765 | Val Micro-F1: 0.9682 | Val Macro-F1: 0.6474 | LR: 1.00e-05
  No Macro-F1 improvement (1/5)
Epoch 04 | Train Loss: 0.0725 | Val Micro-F1: 0.9691 | Val Macro-F1: 0.6492 | LR: 1.00e-05
  No Macro-F1 improvement (2/5)
Epoch 05 | Train Loss: 0.0683 | Val Micro-F1: 0.9690 | Val Macro-F1: 0.6518 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.6518 | Micro-F1: 0.9690
Epoch 06 | Train Loss: 0.0643 | Val Micro-F1: 0.9685 | Val Macro-F1: 0.6568 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.6568 | Micro-F1: 0.9685
Epoch 07 | Train Loss: 0.0617 | Val Micro-F1: 0.9693 | Val Macro-F1: 0.6570 | LR: 1.00e-05
  ✓ Saved best model | Macro-F1: 0.6570 | Micro-F1: 0.9693
Epoc

In [269]:
teacher = model

In [270]:
student_cfg = LMConfig(
vocab_size=VOCAB_SIZE,
d_model=MODEL_DIM,
n_layers=6,
n_heads=N_HEADS,
n_kv_heads=N_KV_HEADS,
d_ff=MODEL_FF,
max_seq_len=MAX_SEQ_LEN,
dropout=0.1,
pos_emb_type="rope",
)
student_encoder_model = EncoderLM(student_cfg)
student_encoder_model = student_encoder_model.to(device)

In [271]:
import re
import torch

def initialize_student_from_teacher(
    student,
    pretrained_state,
    selected_layers=(0, 2, 4, 6, 8, 9),
):
    student_state = student.state_dict()
    copied = []
    skipped = []

    # Copy non-block parameters when names and shapes match.
    for name, target_tensor in student_state.items():
        match = re.match(r"blocks\.(\d+)\.(.*)", name)

        if match:
            continue

        source_tensor = pretrained_state.get(name)

        if (
            source_tensor is not None
            and source_tensor.shape == target_tensor.shape
        ):
            student_state[name] = source_tensor
            copied.append(name)
        else:
            skipped.append(name)

    # Map selected pretrained blocks to student blocks.
    for student_idx, teacher_idx in enumerate(selected_layers):
        student_prefix = f"blocks.{student_idx}."
        teacher_prefix = f"blocks.{teacher_idx}."

        for name, target_tensor in student_state.items():
            if not name.startswith(student_prefix):
                continue

            suffix = name[len(student_prefix):]
            source_name = teacher_prefix + suffix
            source_tensor = pretrained_state.get(source_name)

            if (
                source_tensor is not None
                and source_tensor.shape == target_tensor.shape
            ):
                student_state[name] = source_tensor
                copied.append(name)
            else:
                skipped.append(name)

    student.load_state_dict(student_state)

    print(f"Copied tensors: {len(copied)}")
    print(f"Skipped or unmatched tensors: {len(skipped)}")

    if skipped:
        print("First unmatched names:", skipped[:15])

    return student


student_encoder_model = initialize_student_from_teacher(
    student_encoder_model,
    new_state_dict,
    selected_layers=(0, 2, 4, 6, 8, 9),
).to(device)

In [272]:
# class NERHead(nn.Module):
#     def __init__(self, model, d_model, num_classes):
#         super().__init__()
#         self.model = model
#         self.head = nn.Linear(d_model, num_classes)
    
#     def  get_hidden_states(self, idx, padding_mask=None):
#         B, T = idx.shape
#         # Token embeddings
#         x = self.model.tok_emb(idx)
#         # Sinusoidal positional encoding if Phase-1 used it
#         if self.model.cfg.pos_emb_type == "sinusoidal":
#             x = x + self.model.pos_emb[:T].unsqueeze(0)
#         # Build attention mask
#         if padding_mask is not None:
#             if padding_mask.shape != (B, T):
#                 raise ValueError(
#                     "padding_mask must have shape (B, T)"
#                 )
#             padding_mask = padding_mask.to(
#                 device=idx.device,
#                 dtype=torch.bool
#             )
#         # Encoder is bidirectional
#         is_causal = torch.zeros(
#             B,
#             dtype=torch.bool,
#             device=idx.device
#         )
#         attn_mask = build_attn_mask(
#             is_causal,
#             padding_mask,
#             T
#         )
#         for block in self.model.blocks:
#                 x = block(
#                     x,
#                     attn_mask=attn_mask
#                 )
#         x = self.model.ln_f(x)
#         return x
#     def forward(self, input_ids, padding_mask=None):
#         hidden = self.get_hidden_states(
#             input_ids,
#             padding_mask=padding_mask
#         )
#         logits = self.head(hidden)
#         return logits

In [273]:
# student = NERHead(
#     model=student_encoder_model,
#     d_model=MODEL_DIM,
#     num_classes=NUM_LABELS
# ).to(device)

# # Unfreeze every parameter
# for param in student.parameters():
#     param.requires_grad = True
# if torch.cuda.device_count() > 1:
#     print("Using", torch.cuda.device_count(), "GPUs")
#     student = torch.nn.DataParallel(student)
# student = student.to(device)


# optimizer = torch.optim.AdamW(
#     student.parameters(),
#     lr=1e-4,
#     weight_decay=0.01
# )

# history = train_model_distillation(
#     student,
#     teacher,
#     train_loader,
#     val_loader,
#     optimizer,
#     criterion,
#     epochs=30,
#     patience=3,
#     min_delta=1e-4,
#     max_grad_norm=1.0,
#     use_amp=True,
#     temperature=2.0,
#     alpha=0.5,
#     save_path="best_distilled_ner.pt",
# )

In [274]:
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support

def evaluate_macro_f1(predictions, gold, entity_types):
    """
    Strict entity-level Macro-F1.

    predictions: {doc_id: [{"label": ..., "start": ..., "end": ...}, ...]}
    gold:        {doc_id: [{"label": ..., "start": ..., "end": ...}, ...]}
    entity_types: list of entity labels, e.g. APP, RESP, COURT, ...

    A prediction is correct only if label, start, and end match exactly.
    """

    y_true = []
    y_pred = []

    all_doc_ids = set(gold) | set(predictions)

    for doc_id in all_doc_ids:
        gold_entities = gold.get(doc_id, [])
        pred_entities = predictions.get(doc_id, [])

        gold_set = {
            (e["label"], e["start"], e["end"])
            for e in gold_entities
        }

        pred_set = {
            (e["label"], e["start"], e["end"])
            for e in pred_entities
        }

        # Exact entity matches are true positives.
        matched = gold_set & pred_set

        # Encode entity instances for micro-style matching,
        # then calculate per-entity-type metrics below.
        for label in entity_types:
            tp = sum(1 for e in matched if e[0] == label)
            fp = sum(1 for e in pred_set - gold_set if e[0] == label)
            fn = sum(1 for e in gold_set - pred_set if e[0] == label)

            y_true.extend([label] * tp + [f"__FN__{label}"] * fn)
            y_pred.extend([label] * tp + [f"__FP__{label}"] * fp)

    # Compute per-type scores directly from TP, FP and FN.
    f1_scores = []

    for label in entity_types:
        tp = fp = fn = 0

        for doc_id in all_doc_ids:
            g = {
                (e["label"], e["start"], e["end"])
                for e in gold.get(doc_id, [])
            }
            p = {
                (e["label"], e["start"], e["end"])
                for e in predictions.get(doc_id, [])
            }

            tp += sum(1 for e in g & p if e[0] == label)
            fp += sum(1 for e in p - g if e[0] == label)
            fn += sum(1 for e in g - p if e[0] == label)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        f1_scores.append(f1)

        print(
            f"{label:12s} "
            f"P={precision:.4f} "
            f"R={recall:.4f} "
            f"F1={f1:.4f}"
        )

    macro_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0.0

    print(f"\nStrict Entity Macro-F1: {macro_f1:.4f}")
    return macro_f1

In [275]:
@torch.no_grad()
def predict_document(
    model,
    text,
    tokenizer,
    pad_id,
    device,
    max_len=1024
):
    model.eval()

    # ---------------------------------------------------------
    # Tokenize entire document
    # ---------------------------------------------------------

    token_ids, offsets = encode_with_offsets(text)

    n_tokens = len(token_ids)

    # Store prediction for every original token
    all_pred_ids = [None] * n_tokens

    # ---------------------------------------------------------
    # Process chunks
    # ---------------------------------------------------------

    for start in range(0, n_tokens, max_len):

        end = min(
            start + max_len,
            n_tokens
        )

        chunk_ids = token_ids[start:end]

        input_ids = torch.tensor(
            [chunk_ids],
            dtype=torch.long,
            device=device
        )

        attention_mask = torch.ones(
            (1, len(chunk_ids)),
            dtype=torch.long,
            device=device
        )

        padding_mask = ~attention_mask.bool()

        # -----------------------------------------------------
        # Model
        # -----------------------------------------------------

        logits = model(
            input_ids,
            padding_mask=padding_mask
        )

        pred_ids = logits.argmax(dim=-1)[0]

        pred_ids = pred_ids.cpu().tolist()

        # -----------------------------------------------------
        # Put predictions back into document positions
        # -----------------------------------------------------

        for local_idx, pred_id in enumerate(pred_ids):

            global_idx = start + local_idx

            all_pred_ids[global_idx] = pred_id

    # ---------------------------------------------------------
    # Convert IDs → BIO labels
    # ---------------------------------------------------------

    pred_labels = [
        id2label[pred_id]
        for pred_id in all_pred_ids
    ]

    return pred_labels, offsets

In [276]:
def bio_to_entities(pred_labels):
    entities = []

    current_label = None
    start = None
    has_i = False

    for i, tag in enumerate(pred_labels):

        if tag == "O":

            if current_label is not None and has_i:
                entities.append((start, i, current_label))

            current_label = None
            start = None
            has_i = False

        elif tag.startswith("B-"):

            # Close previous entity
            if current_label is not None and has_i:
                entities.append((start, i, current_label))

            current_label = tag[2:]
            start = i
            has_i = False

        elif tag.startswith("I-"):

            label = tag[2:]

            if current_label == label:
                has_i = True

            else:
                # Invalid/orphan I:
                # don't start a new entity
                pass

    # Close final entity
    if current_label is not None and has_i:
        entities.append(
            (start, len(pred_labels), current_label)
        )

    return entities

In [277]:
def entities_to_character_spans(
    text,
    token_entities,
    offsets,
    suffix_map
):
    final_entities = []

    for start_token, end_token, label in token_entities:

        char_start = offsets[start_token][0]
        char_end = offsets[end_token - 1][1]

        predicted_text = text[
            char_start:char_end
        ]

        # -----------------------------------------------------
        # Learned suffix correction
        # -----------------------------------------------------

        key = (label, predicted_text)

        if key in suffix_map:

            corrected_text = suffix_map[key]

            # Safety check
            if predicted_text.startswith(
                corrected_text
            ):
                char_end = (
                    char_start
                    + len(corrected_text)
                )

        # -----------------------------------------------------
        # Final text
        # -----------------------------------------------------

        entity_text = text[
            char_start:char_end
        ]

        final_entities.append([
            char_start,
            char_end,
            label,
            entity_text
        ])

    return final_entities

In [278]:
def predict_test_document(
    model,
    doc_id,
    test_text,
    tokenizer,
    pad_id,
    device,
    suffix_map,
    max_len=1024
):
    text = test_text[doc_id]

    # ---------------------------------------------------------
    # Model prediction
    # ---------------------------------------------------------

    pred_labels, offsets = predict_document(
        model=model,
        text=text,
        tokenizer=tokenizer,
        pad_id=pad_id,
        device=device,
        max_len=max_len
    )

    # ---------------------------------------------------------
    # BIO → token entities
    # ---------------------------------------------------------

    token_entities = bio_to_entities(
        pred_labels
    )

    # ---------------------------------------------------------
    # Token → character + suffix correction
    # ---------------------------------------------------------

    final_entities = entities_to_character_spans(
        text=text,
        token_entities=token_entities,
        offsets=offsets,
        suffix_map=suffix_map
    )

    return final_entities

In [279]:
def apply_prefix_correction(entities, prefix_map):
    corrected = []

    for entity in entities:
        entity = list(entity)

        label, text = entity[0], entity[1]
        key = (label, text)

        if key in prefix_map:
            entity[1] = prefix_map[key]

        corrected.append(entity)

    return corrected

In [280]:
# test_predictions = {}

# for doc_id, text in fold3_text.items():
#     entities = predict_test_document(
#         model=model,
#         doc_id=doc_id,
#         test_text=fold3_text,
#         tokenizer=converter,
#         pad_id=0,
#         device=device,
#         suffix_map=suffix_map,
#         max_len=1024
#     )

#     entities = apply_prefix_correction(entities, prefix_map)
#     test_predictions[doc_id] = entities

# print("Documents predicted:", len(test_predictions))

In [281]:
test_predictions = {}

for doc_id, text in fold3_text.items():

    entities = predict_test_document(
        model=model,
        doc_id=doc_id,
        test_text=fold3_text,
        tokenizer=converter,
        pad_id=0,
        device=device,
        suffix_map=suffix_map,
        max_len=1024
    )

    test_predictions[doc_id] = entities

print(
    "Documents predicted:",
    len(test_predictions)
)

Documents predicted: 35


In [282]:
import json

# Remove entity text; keep only [start, end, label]
test_predictions_spans = {
    doc_id: [
        [start, end, label]
        for start, end, label, entity_text in entities
    ]
    for doc_id, entities in test_predictions.items()
}

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions_spans,
        f,
        ensure_ascii=False,
        indent=2
    )

In [283]:
import os
import zipfile

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

In [284]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)

Codabench submission created:
/kaggle/working/cs781_phase2/submission.zip


In [285]:
import os
import shutil
from IPython.display import display, FileLink

# Copy to Kaggle's working directory with a simple filename
download_path = "/kaggle/working/submission.zip"

shutil.copy2(SUBMISSION_ZIP, download_path)

print("Exists:", os.path.isfile(download_path))
print("Size:", os.path.getsize(download_path), "bytes")

display(FileLink("submission.zip"))

Exists: True
Size: 28618 bytes


/kaggle/working/submission.zip